In [18]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    
    # 1. Use ProactorEventLoop for subprocess support
    # Problem: Windows has two event loop types:
    #   - SelectorEventLoop (default) - doesn't support subprocesses well
    #   - ProactorEventLoop - properly supports subprocesses and pipes
    # Solution: Switch to ProactorEventLoop if not already using it
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    # Problem: In Jupyter/IPython, sys.stderr is wrapped in a special object
    #   that doesn't have a real file descriptor (fileno). When subprocess
    #   tries to use it, it crashes with "io.UnsupportedOperation: fileno"
    # Solution: Replace Jupyter's custom stderr with the original system stderr
    #   which has a proper file descriptor
    # Check: Only do this if running inside IPython/Jupyter kernel
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__

## Local MCP server

In [19]:
import sys
import os

# Get the Python executable from your current environment
python_executable = sys.executable
# print(f"Using Python: {python_executable}")

script_path = os.path.abspath("resources/Mcp_Server.py")

from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
            "transport": "stdio",
            "command": python_executable,  # Use the current environment's Python
            "args": [script_path],
        }
    }
)

In [20]:
# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("local_server")

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

In [21]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model_provider="ollama",
    model="qwen3:8b"
)

from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=prompt
)

In [22]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config
)

In [23]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Tell me about the langchain-mcp-adapters library', additional_kwargs={}, response_metadata={}, id='5b602b41-4d21-4af5-a44c-cf37ede893f2'),
              AIMessage(content='<think>\nOkay, the user is asking about the langchain-mcp-adapters library. First, I need to figure out what this library is. I know that LangChain is a framework for developing applications using large language models, but I\'m not familiar with the "mcp-adapters" part. Maybe it\'s a specific adapter or integration within the LangChain ecosystem.\n\nSince I don\'t have prior knowledge about this specific library, I should use the search_web tool to look for information. The user might be referring to a package that connects LangChain with another system or service, perhaps MCP standing for something like Machine Control Protocol or another acronym. Alternatively, it could be a typo or a less-known component.\n\nI\'ll start by searching the web for "langchain-mcp-adapters" to see i

## Online MCP

In [24]:
import os

# Build the full path to uvx.exe
conda_env = os.path.dirname(os.path.dirname(os.__file__))
uvx_path = os.path.join(conda_env, "Scripts", "uvx.exe")
# print(f"uvx.exe path: {uvx_path}")
print(f"Exists: {os.path.exists(uvx_path)}")

client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": uvx_path,  # Use full path to uvx.exe
            "args": [
                "mcp-server-time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client.get_tools()

Exists: True


In [25]:
agent = create_agent(
    model=model,
    tools=tools,
)

In [26]:
question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)

{'messages': [HumanMessage(content='What time is it?', additional_kwargs={}, response_metadata={}, id='9b5796d3-93d9-4d7e-bba3-5f06e1e87bbf'),
              AIMessage(content='<think>\nOkay, the user asked, "What time is it?" So I need to figure out the current time. Let me check the tools available. There\'s a function called get_current_time that retrieves the current time in a specific timezone. The parameters require a timezone, and if the user doesn\'t specify, it should use \'America/New_York\' as the local timezone.\n\nSince the user didn\'t mention any particular timezone, I should assume they want the time in their local timezone, which according to the tool\'s description is \'America/New_York\' by default. So I need to call get_current_time with the timezone set to \'America/New_York\'. That should give the current time in the correct location. Let me make sure I\'m using the right function and parameters. Yep, that\'s right. No other functions are needed here. Just get the 

## Travel Agent

In [36]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "travel_server": {
                "transport": "streamable_http",
                "url": "https://mcp.kiwi.com"
            }
    }
)

tools = await client.get_tools()

In [37]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model,
    tools=tools,
    checkpointer=InMemorySaver(),
    system_prompt="You are a travel agent. No follow up questions."
)

In [38]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Get me a direct flight from San Francisco to Tokyo on March 31st 2026")]},
    config
    )

In [39]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Get me a direct flight from San Francisco to Tokyo on March 31st 2026', additional_kwargs={}, response_metadata={}, id='ec2d72ff-302c-4da3-9532-9fae90813be5'),
              AIMessage(content='<think>\nOkay, the user wants a direct flight from San Francisco to Tokyo on March 31st, 2026. Let me start by checking the available functions. The search-flight function seems appropriate here.\n\nFirst, I need to extract the parameters. The flyFrom is San Francisco, flyTo is Tokyo. The departureDate is March 31st, 2026, which in dd/mm/yyyy format is 31/03/2026. Since they specified a direct flight, I should ensure that the function parameters include that. Wait, looking at the function\'s parameters, there\'s no explicit \'direct\' parameter. Hmm, maybe the API handles that by default, or perhaps I need to set the layovers to zero? Wait, the function\'s description mentions that the tool will resolve airport codes and find available flights. So maybe the API

In [40]:
print(response["messages"][-1].content)

<think>
Okay, so I need to help the user find the best flight from San Francisco to Tokyo based on the data provided. Let me start by understanding the data. The user has given a list of flight options with various details like departure and arrival times, durations, prices, and layovers. 

First, I should check the key factors that matter for a flight: price, duration, number of layovers, and the time of departure and arrival. The user might be looking for the cheapest option, the fastest, or something in between. Since the user hasn't specified, I should consider all these aspects and present the options accordingly.

Looking at the prices, the cheapest flights seem to be around 362 EUR. There are a couple of flights priced at 362 and 363 EUR. The more expensive ones are around 437 and 437 EUR. So, the budget-friendly options are the ones with the lower prices. 

Next, the duration. The shortest flight is 9 hours and 30 minutes (like the one with 95700 seconds), but that's with a lay